In [2]:
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append("..")

from src.utils import *

## 

The goal of this research project is to be able to extract the dyamical critical exponent from a multi-species exclusion process using just Topological Data Analysis. Note that 

In [3]:
rates_matrix = np.array([[0.0, 1.0, 1.0], [0.0, 0.0, 1.0], [0.0, 0.0, 0.0]], dtype=np.float64) # totally asymmetric 
#rates_matrix = np.array([[0.0, 1.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 0.0]], dtype=np.float64) # symmetric

process = MultiSpeciesExclusionProcess(dimension=3, density=[1/3, 1/3, 1/3], rates_matrix=rates_matrix, length=102, shuffle=True)

In [4]:
H = process.normal_mode_height_time_series(n_samples=1000, sample_every=1)

In [5]:
h_0 = H[:, :, 0]
h_1 = H[:, :, 1]

In [ ]:
import os
import numpy as np
import gudhi as gd
import git_root
import matplotlib.pyplot as plt

from concurrent.futures import ProcessPoolExecutor
from scipy.signal import savgol_filter

from src.utils import *


# ============================================================
# Lightweight point-cloud construction
# ============================================================

def mode_patch_point_cloud(
    H,
    mode_index,
    time_start,
    time_window=8,
    space_window=5,
    stride=4,
    max_points=400,
    standardize=True,
    seed=0,
):
    """
    Build a smaller point cloud from local spatial patches.

    Main memory/CPU saver:
        - stride > 1
        - smaller time_window
        - smaller space_window
        - random subsampling to max_points
    """

    h = H[:, :, mode_index]

    time_end = min(time_start + time_window, h.shape[0])
    block = h[time_start:time_end, :]

    points = []

    for t in range(block.shape[0]):
        for x in range(0, block.shape[1] - space_window + 1, stride):
            patch = block[t, x:x + space_window]
            points.append(patch)

    point_cloud = np.array(points, dtype=np.float64)

    if point_cloud.shape[0] > max_points:
        rng = np.random.RandomState(seed)
        idx = rng.choice(point_cloud.shape[0], size=max_points, replace=False)
        point_cloud = point_cloud[idx]

    if standardize and point_cloud.size > 0:
        mean = np.mean(point_cloud, axis=0)
        std = np.std(point_cloud, axis=0)
        std[std == 0.0] = 1.0
        point_cloud = (point_cloud - mean) / std

    return point_cloud


# ============================================================
# TDA feature functions
# ============================================================

def persistence_diagram(point_cloud, max_edge_length=2.0):
    point_cloud = np.asarray(point_cloud, dtype=np.float64)
    point_cloud = np.unique(point_cloud, axis=0)

    if point_cloud.shape[0] < 3:
        return np.empty((0, 2)), np.empty((0, 2))

    rips = gd.RipsComplex(points=point_cloud, max_edge_length=max_edge_length)
    simplex_tree = rips.create_simplex_tree(max_dimension=2)
    simplex_tree.persistence()

    h0 = simplex_tree.persistence_intervals_in_dimension(0)
    h1 = simplex_tree.persistence_intervals_in_dimension(1)

    if h0.shape[0] > 0:
        h0 = h0[np.isfinite(h0[:, 1])]

    if h1.shape[0] > 0:
        h1 = h1[np.isfinite(h1[:, 1])]

    return h0, h1


def betti_curve(intervals, r_values):
    if intervals.shape[0] == 0:
        return np.zeros(len(r_values), dtype=np.float64)

    births = intervals[:, 0]
    deaths = intervals[:, 1]

    values = []
    for r in r_values:
        values.append(np.sum((births <= r) & (r < deaths)))

    return np.array(values, dtype=np.float64)


def tda_feature_vector(point_cloud, r_values, epsilon=0.2, max_edge_length=2.0):
    h0, h1 = persistence_diagram(point_cloud, max_edge_length=max_edge_length)

    beta0_curve = betti_curve(h0, r_values)
    beta1_curve = betti_curve(h1, r_values)

    if h1.shape[0] > 0:
        persistences = h1[:, 1] - h1[:, 0]
        p_max = np.max(persistences)
        p_total = np.sum(persistences)
        n_epsilon = np.sum(persistences > epsilon)
    else:
        p_max = 0.0
        p_total = 0.0
        n_epsilon = 0.0

    return np.concatenate(
        [
            beta0_curve,
            beta1_curve,
            np.array([p_max, p_total, n_epsilon], dtype=np.float64),
        ]
    )


# ============================================================
# TDA time series for one mode
# ============================================================

def mode_tda_time_series(
    H,
    mode_index,
    r_values,
    time_window=8,
    time_step=20,
    space_window=5,
    stride=4,
    max_points=400,
    epsilon=0.2,
    max_edge_length=2.0,
    seed=0,
):
    n_samples = H.shape[0]

    features = []
    window_times = []

    for k, time_start in enumerate(range(0, n_samples - time_window + 1, time_step)):
        pc = mode_patch_point_cloud(
            H=H,
            mode_index=mode_index,
            time_start=time_start,
            time_window=time_window,
            space_window=space_window,
            stride=stride,
            max_points=max_points,
            standardize=True,
            seed=seed + 1000 * mode_index + k,
        )

        fv = tda_feature_vector(
            point_cloud=pc,
            r_values=r_values,
            epsilon=epsilon,
            max_edge_length=max_edge_length,
        )

        features.append(fv)
        window_times.append(time_start + time_window // 2)

    return np.array(window_times), np.array(features)


# ============================================================
# Relaxation-time extraction
# ============================================================

def tda_relaxation_curve(F, steady_fraction=0.2):
    F = np.asarray(F, dtype=np.float64)

    n = F.shape[0]
    steady_start = int((1.0 - steady_fraction) * n)

    F_steady = np.mean(F[steady_start:], axis=0)
    Q = np.linalg.norm(F - F_steady, axis=1)

    return Q


def smooth_curve(y, window_length=9, polyorder=3):
    y = np.asarray(y, dtype=np.float64)

    if len(y) < 5:
        return y

    if window_length >= len(y):
        window_length = len(y) - 1

    if window_length % 2 == 0:
        window_length -= 1

    if window_length <= polyorder:
        return y

    return savgol_filter(y, window_length=window_length, polyorder=polyorder)


def relaxation_time_from_decay(
    times,
    Q,
    threshold_fraction=0.3,
    steady_fraction=0.2,
    window_fraction=0.05,
    required_fraction=0.8,
):
    times = np.asarray(times, dtype=np.float64)
    Q = np.asarray(Q, dtype=np.float64)

    n = len(Q)

    steady_start = int((1.0 - steady_fraction) * n)
    baseline = np.mean(Q[steady_start:])

    initial = np.mean(Q[:max(3, n // 20)])
    amplitude = initial - baseline

    if amplitude <= 0:
        return np.nan, baseline

    threshold = baseline + threshold_fraction * amplitude
    window = max(3, int(window_fraction * n))

    for i in range(0, n - window):
        frac_below = np.mean(Q[i:i + window] <= threshold)

        if frac_below >= required_fraction:
            return times[i], threshold

    return np.nan, threshold


# ============================================================
# One simulation run
# ============================================================

def single_run_mode_tda_relaxation_curves(
    L,
    rates_matrix,
    density,
    n_samples,
    sample_every,
    r_values,
    time_window,
    time_step,
    space_window,
    stride,
    max_points,
    epsilon,
    max_edge_length,
    seed,
):
    np.random.seed(seed)

    model = MultiSpeciesExclusionProcess(
        dimension=len(density),
        density=density,
        rates_matrix=rates_matrix,
        length=L,
        shuffle=False,
    )

    model.set_chain(equal_spread_chain(L).tolist())

    H = model.normal_mode_height_time_series(
        n_samples=n_samples,
        sample_every=sample_every,
    )

    n_modes = H.shape[2]

    Q_modes = []
    physical_times = None

    for mode_index in range(n_modes):
        window_times, F = mode_tda_time_series(
            H=H,
            mode_index=mode_index,
            r_values=r_values,
            time_window=time_window,
            time_step=time_step,
            space_window=space_window,
            stride=stride,
            max_points=max_points,
            epsilon=epsilon,
            max_edge_length=max_edge_length,
            seed=seed,
        )

        Q = tda_relaxation_curve(F)
        Q_modes.append(Q)

        if physical_times is None:
            physical_times = window_times * sample_every

    return physical_times, np.array(Q_modes)


# ============================================================
# Fit dynamic exponent
# ============================================================

def fit_dynamic_exponent(L_values, tau_values):
    L_values = np.asarray(L_values, dtype=np.float64)
    tau_values = np.asarray(tau_values, dtype=np.float64)

    valid = np.isfinite(tau_values) & (tau_values > 0)

    log_L = np.log(L_values[valid])
    log_tau = np.log(tau_values[valid])

    if len(log_L) < 2:
        return np.nan, np.nan, log_L, log_tau

    z, intercept = np.polyfit(log_L, log_tau, 1)

    return z, intercept, log_L, log_tau


def classify_universality(z):
    if not np.isfinite(z):
        return "unknown"

    if abs(z - 1.5) < abs(z - 2.0):
        return "KPZ-like"
    else:
        return "diffusive/EW-like"


# ============================================================
# Main experiment
# ============================================================

if __name__ == "__main__":

    density = [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0]

    rates_matrix = np.array(
        [
            [0.0, 1.0, 1.0],
            [0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0],
        ],
        dtype=np.float64,
    )

    # Smaller test sweep first.
    L_values = np.array([60, 90, 120, 150, 180, 210], dtype=int)

    N_runs = 8

    n_samples = 700
    sample_every = 200

    r_values = np.linspace(0.1, 2.5, 30)

    time_window = 12
    time_step = 10
    space_window = 6
    stride = 2
    max_points = 900

    epsilon = 0.2
    max_edge_length = 2.5

    threshold_fraction = 0.5
    steady_fraction = 0.2
    window_fraction = 0.05

    root = git_root.git_root()
    output_dir = f"{root}/data/mode_separated_tda_dce_light"
    os.makedirs(output_dir, exist_ok=True)

    n_modes = len(density) - 1
    tau_by_mode = [[] for _ in range(n_modes)]

    print(f"using max_workers = {max_workers}")
    print(f"L_values = {L_values}")
    print(f"N_runs = {N_runs}")
    print(f"n_samples = {n_samples}, sample_every = {sample_every}")
    print(f"time_window = {time_window}, time_step = {time_step}")
    print(f"space_window = {space_window}, stride = {stride}, max_points = {max_points}")

    for L in L_values:
        print()
        print(f"running L = {L}")

        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            futures = []

            for run in range(N_runs):
                seed = 1000000 * int(L) + run

                futures.append(
                    executor.submit(
                        single_run_mode_tda_relaxation_curves,
                        L,
                        rates_matrix,
                        density,
                        n_samples,
                        sample_every,
                        r_values,
                        time_window,
                        time_step,
                        space_window,
                        stride,
                        max_points,
                        epsilon,
                        max_edge_length,
                        seed,
                    )
                )

            results = [f.result() for f in futures]

        physical_times = results[0][0]

        Q_ensemble = np.array([res[1] for res in results])
        Q_mean_by_mode = np.mean(Q_ensemble, axis=0)

        for mode_index in range(n_modes):
            Q_mean = Q_mean_by_mode[mode_index]
            Q_smooth = smooth_curve(Q_mean, window_length=9, polyorder=3)

            tau, threshold = relaxation_time_from_decay(
                physical_times,
                Q_smooth,
                threshold_fraction=threshold_fraction,
                steady_fraction=steady_fraction,
                window_fraction=window_fraction,
            )

            tau_by_mode[mode_index].append(tau)

            print(
                f"mode {mode_index}: tau = {tau}, "
                f"threshold = {threshold}"
            )

        plt.figure(figsize=(8, 5))

        for mode_index in range(n_modes):
            Q_mean = Q_mean_by_mode[mode_index]
            Q_smooth = smooth_curve(Q_mean, window_length=9, polyorder=3)
            tau = tau_by_mode[mode_index][-1]

            plt.plot(
                physical_times,
                Q_smooth,
                linewidth=2,
                label=fr"mode {mode_index}, $\tau={tau:.0f}$",
            )

            if np.isfinite(tau):
                plt.axvline(tau, linestyle="--", alpha=0.5)

        plt.xlabel("Monte Carlo time")
        plt.ylabel("TDA distance from steady state")
        plt.title(f"Mode-separated TDA relaxation, L={L}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{output_dir}/tda_relaxation_L_{L}.png", dpi=300)
        plt.close()

    z_values = []
    intercepts = []
    logLs = []
    logTaus = []

    for mode_index in range(n_modes):
        tau_values = np.array(tau_by_mode[mode_index], dtype=np.float64)

        z, intercept, log_L, log_tau = fit_dynamic_exponent(
            L_values,
            tau_values,
        )

        z_values.append(z)
        intercepts.append(intercept)
        logLs.append(log_L)
        logTaus.append(log_tau)

        print()
        print(f"mode {mode_index}")
        print(f"  tau values = {tau_values}")
        print(f"  z = {z}")
        print(f"  class = {classify_universality(z)}")

    plt.figure(figsize=(7, 5))

    for mode_index in range(n_modes):
        z = z_values[mode_index]
        intercept = intercepts[mode_index]
        log_L = logLs[mode_index]
        log_tau = logTaus[mode_index]

        if len(log_L) < 2:
            continue

        fit_line = z * log_L + intercept

        plt.scatter(
            log_L,
            log_tau,
            alpha=0.75,
            label=fr"mode {mode_index}, $z={z:.3f}$",
        )

        plt.plot(
            log_L,
            fit_line,
            linestyle="--",
            linewidth=2,
        )

    plt.xlabel(r"$\log L$", fontsize=14)
    plt.ylabel(r"$\log \tau(L)$", fontsize=14)
    plt.title("Mode-separated TDA dynamic exponents", fontsize=16)

    plt.grid(True, linestyle=":", linewidth=1)
    plt.legend(fontsize=12)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/mode_separated_tda_dynamic_exponents.png", dpi=300)
    plt.show()

using max_workers = 1
L_values = [ 60  90 120 150 180 210]
N_runs = 8
n_samples = 700, sample_every = 200
time_window = 12, time_step = 10
space_window = 6, stride = 2, max_points = 900

running L = 60
mode 0: tau = 23200.0, threshold = 110.12917105759266
mode 1: tau = 3200.0, threshold = 105.95599226318747

running L = 90
mode 0: tau = 29200.0, threshold = 147.3504441870341
mode 1: tau = nan, threshold = 141.56795743059644

running L = 120
mode 0: tau = nan, threshold = 158.65480173546032
mode 1: tau = 5200.0, threshold = 174.65930642792742

running L = 150
mode 0: tau = 95200.0, threshold = 191.26818690706818
mode 1: tau = 23200.0, threshold = 201.5939639915954

running L = 180
mode 0: tau = 5200.0, threshold = 195.25534077015982
mode 1: tau = 5200.0, threshold = 211.4816794968579

running L = 210
